In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

In [2]:
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to C:\Users\lenovo
[nltk_data]     2020\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\lenovo
[nltk_data]     2020\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
data = pd.read_csv('Reviews.csv')

In [4]:
data.head()

,Content,Url,Date
0,"""Power and diverse, Yes! | Complete all in o...",https://www.g2.com/survey_responses/hubspot-ma...,2023-01-01
1,"""Hubspot Is More Than Just A 2-in-1 Shampoo Bo...",https://www.g2.com/survey_responses/hubspot-ma...,2023-01-01
2,"""Hubspot delivers more value than most of our ...",https://www.g2.com/survey_responses/hubspot-ma...,2023-01-01
3,"""All-in-one marketing, sales, and customer ser...",https://www.g2.com/survey_responses/hubspot-ma...,2023-01-01
4,"""Great overall experience using HubSpot Market...",https://www.g2.com/survey_responses/hubspot-ma...,2023-01-01


In [5]:
def clean_text(text):
    text = re.sub(r'Review collected by and hosted on G2.com', '', text)
    text = text.strip().replace('"', '')  # Remove extraneous characters
    return text

In [6]:
data['Cleaned_Content'] = data['Content'].apply(clean_text)

In [7]:
def assign_sentiment(text):
    positive_words = ["best", "great", "love", "fantastic", "excellent", "game-changer", "perfect", "good"]
    negative_words = ["difficult", "lack", "problem", "bad", "poor", "issue", "hard", "dislike"]
    text_tokens = word_tokenize(text.lower())
    
    # Count positive and negative words
    pos_count = sum(1 for word in text_tokens if word in positive_words)
    neg_count = sum(1 for word in text_tokens if word in negative_words)
    
    # Assign label based on counts
    return 1 if pos_count > neg_count else 0

In [8]:
data['Sentiment'] = data['Cleaned_Content'].apply(assign_sentiment)

In [9]:
stop_words = set(stopwords.words('english'))

In [10]:
def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    return ' '.join(tokens)

In [11]:
data['Processed_Content'] = data['Cleaned_Content'].apply(preprocess_text)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    data['Processed_Content'], data['Sentiment'], test_size=0.2, random_state=42)

In [13]:
tfidf = TfidfVectorizer(max_features=1000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [14]:
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

LogisticRegression()

In [15]:
y_pred = model.predict(X_test_tfidf)

In [16]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.7545454545454545
Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.57      0.64        42
           1       0.77      0.87      0.81        68

    accuracy                           0.75       110
   macro avg       0.75      0.72      0.73       110
weighted avg       0.75      0.75      0.75       110



In [17]:
def predict_sentiment(new_text):
    cleaned_text = preprocess_text(clean_text(new_text))
    text_tfidf = tfidf.transform([cleaned_text])
    return "Positive" if model.predict(text_tfidf)[0] == 1 else "Negative"

In [18]:
new_review = "This tool is really effective and easy to use!"
print("New Review Sentiment:", predict_sentiment(new_review))

New Review Sentiment: Positive
